In [ ]:
import uproot
import os
import mplhep as hep
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pathlib

from scipy.stats import spearmanr
from source.importer import nanoaod_to_dataframe, get_z_m_pt, initialize_dir, require_min_n, read_event_list
from source.genmatching import calculate_dr, apply_genmatching, remove_muon_jets, remove_nonmatches, find_unmatchable_objects, remove_zmumu_candidates, dist_between_zmumu_unmatch, get_sub_df, remove_non_muon_jets
from source.helper import verify_events, create_concordant_subsets, copy_columns_from_to, get_matching_df, subtract_columns, prepare_matching, get_n_occurence, set_working_dir, count_n_objects, copy_column_set, print_stats, print_zmumu_stats, set_plt_fonts
from source.plotting import match_plot, control_plot, nq_comparison, x_vs_y

from source.importer import quality_cut, assert_object_validity, compactify_objects, get_jet_basenames, get_muon_basenames, get_electron_basenames


In [ ]:
########################################################################################################################################################################
# paths for input and output 
########################################################################################################################################################################


# data_path = "./data/2022G-nanoaod_gen/"
# emb_path = "./data/2022G-nanoaod_gen/"
# data_path = "./data/2022_patched_driver/"
# emb_path = "./data/2022_patched_driver/"
data_path = "./data/2022_nofsr/"
emb_path = "./data/2022_nofsr/"


data_filenames = "2022G-data_*.root"
emb_filenames = "2022G-embedding_*.root"

# selection_file = os.path.join(data_path, "selection.csv")
set_working_dir()
set_plt_fonts()

output_path = "./output_nofsr_all/unmatched/"
initialize_dir(output_path, purge=False)

muon_plots = True
jet_plots = True
electron_plots = False
photon_plots = False

muon_cut = 0.1
jet_cut = 0.2

# electron_cut = 0.3
# photon_cut = 0.3

In [ ]:
########################################################################################################################################################################
# columns to be read and their new names in the resulting df
########################################################################################################################################################################



data_quantities = [
    {"key":"PuppiMET_pt",       "target":"PuppiMET_pt",     "expand":False},
    {"key":"PuppiMET_phi",      "target":"PuppiMET_phi",    "expand":False},
    {"key":"Muon_phi",          "target":"Muon_phi",        "expand":True},
    {"key":"Muon_pt",           "target":"Muon_pt",         "expand":True},
    {"key":"Muon_eta",          "target":"Muon_eta",        "expand":True},
    {"key":"Muon_mass",         "target":"Muon_m",          "expand":True},
    {"key":"Jet_phi",           "target":"Jet_phi",         "expand":True},
    {"key":"Jet_pt",            "target":"Jet_pt",          "expand":True},
    {"key":"Jet_eta",           "target":"Jet_eta",         "expand":True},
    {"key":"Jet_mass",          "target":"Jet_m",           "expand":True},
    {"key":"run",               "target":"run",             "expand":False},
    {"key":"luminosityBlock",   "target":"lumi",            "expand":False},
    {"key":"event",             "target":"event",           "expand":False},
    {"key":"Muon_isGlobal",     "target":"MuonIsGlobal",    "expand":True},
    {"key":"Muon_tightId",      "target":"MuonIsTight",     "expand":True},
    {"key":"Muon_mediumId",     "target":"MuonIsMedium",    "expand":True},
    {"key":"Muon_looseId",      "target":"MuonIsLoose",     "expand":True},

    # {"key":"Electron_mass",     "target":"Electron_m",      "expand":True},
    # {"key":"Electron_eta",      "target":"Electron_eta",    "expand":True},
    # {"key":"Electron_phi",      "target":"Electron_phi",    "expand":True},
    # {"key":"Electron_pt",       "target":"Electron_pt",     "expand":True},
    # {"key":"Photon_phi",        "target":"Photon_phi",      "expand":True},
    # {"key":"Photon_eta",        "target":"Photon_eta",      "expand":True},
    # {"key":"Photon_pt",         "target":"Photon_pt",       "expand":True},
]

selection_q = [
    {"key":"TauEmbedding_chargeLeadingMuon",		"target":"LM_charge",	"expand":False },
    {"key":"TauEmbedding_chargeTrailingMuon",		"target":"TM_charge",	"expand":False },
    {"key":"TauEmbedding_phiLeadingMuon",		    "target":"LM_phi",		"expand":False },
    {"key":"TauEmbedding_phiTrailingMuon",		    "target":"TM_phi",		"expand":False },
    {"key":"TauEmbedding_ptLeadingMuon",		    "target":"LM_pt",		"expand":False },
    {"key":"TauEmbedding_ptTrailingMuon",		    "target":"TM_pt",		"expand":False },
    {"key":"TauEmbedding_etaLeadingMuon",		    "target":"LM_eta",		"expand":False },
    {"key":"TauEmbedding_etaTrailingMuon",		    "target":"TM_eta",		"expand":False },
    {"key":"TauEmbedding_massLeadingMuon",		    "target":"LM_m",		"expand":False },
    {"key":"TauEmbedding_massTrailingMuon",		    "target":"TM_m",		"expand":False },
    {"key":"nGenPart",		                        "target":"nGenPart",    "expand":False },
    # {"key":"GenPart_pdgId",		                    "target":"GenPart_Id",	"expand":True },
    # {"key":"GenPart_eta",		                    "target":"GenPart_eta",	"expand":True },
    # {"key":"GenPart_phi",		                    "target":"GenPart_phi",	"expand":True },
    # {"key":"GenPart_pt",		                    "target":"GenPart_pt",	"expand":True },
]

emb_quantities = data_quantities.copy()
emb_quantities += selection_q

In [ ]:
########################################################################################################################################################################
# Reading data
########################################################################################################################################################################

print("Loading data")

data_files = list(pathlib.Path(data_path).glob(data_filenames))
emb_files = list(pathlib.Path(emb_path).glob(emb_filenames))

data_df = nanoaod_to_dataframe(files=data_files, quantities=data_quantities)
emb_df = nanoaod_to_dataframe(files=emb_files, quantities=emb_quantities)

data_df, emb_df = create_concordant_subsets(data_df, emb_df)

print(f"Length dataset:\t {len(emb_df)} events")

print("Data loaded")


# selection_df = read_event_list(selection_file)
# data_df, selection_df = create_concordant_subsets(data_df, selection_df)

# print(f"Reduced dataset to {len(selection_df)} events")

In [ ]:
n_jets_data = count_n_objects(data_df, "Jet_eta")
n_jets_emb = count_n_objects(emb_df, "Jet_eta")

np.sum(n_jets_data), np.sum(n_jets_emb)

In [ ]:
if jet_plots:
    n_jets_data = count_n_objects(data_df, "Jet_eta_")
    # n_jets_emb = count_n_objects(emb_df, "Jet_eta_")

    dr = calculate_dr(emb_df.copy(deep=True), "jet_all", filter=None, df2=data_df.copy(deep=True))

    j_data_unmatched, j_emb_unmatched = find_unmatchable_objects(dr, emb_df.copy(deep=True), data_df.copy(deep=True), "jet_all", jet_cut)

    nlone_jets_data = count_n_objects(j_data_unmatched, "Jet_eta_")
    nlone_jets_emb = count_n_objects(j_emb_unmatched, "Jet_eta_")

    n_matchable_jets = n_jets_data - nlone_jets_data
    # n_matchable_jets_emb = n_jets_emb - nlone_jets_emb # is the same

    print_stats(n_matchable_jets, nlone_jets_data, nlone_jets_emb)


    j_data_unmatched = copy_columns_from_to(emb_df, j_data_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
    j_emb_unmatched = copy_columns_from_to(emb_df, j_emb_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
    distances_data, distances_emb = dist_between_zmumu_unmatch(j_data_unmatched, j_emb_unmatched)

    print_zmumu_stats(distances_data, distances_emb, 0.4)


In [ ]:
if jet_plots:
    max_nbins = max([np.amax(nlone_jets_data), np.amax(nlone_jets_emb), np.amax(n_matchable_jets)])
    bins = np.arange(-0.5, max_nbins+0.5, 1)
    ax = nq_comparison({"Matched jets":n_matchable_jets, "Unm. jets in data": nlone_jets_data, "Unm. jets in emb": nlone_jets_emb}, bins, "Number of jets")
    ax.set_yscale("log")

    plt.savefig(os.path.join(output_path, "unmatched_jets_00.png"))
    plt.close()


    bins = np.linspace(0,np.amax([np.nanmax(distances_data), np.nanmax(distances_emb)]), 30)
    ax = nq_comparison({"Data distances":distances_data.flatten(), "Embedding distances":distances_emb.flatten()}, bins, r"$\Delta R_\text{min (Z->µµ, unm. jet)}$")
    ax.vlines(0.4, ax.get_ylim()[0], ax.get_ylim()[1], color="black", ls="dashed", linewidth=2)
    ax.set_yscale("log")
    plt.savefig(os.path.join(output_path, "dr_zmumu_unmatched_jet_00.png"))
    plt.close()

In [ ]:
# nbins= 35

# eta_cols_data = [col for col in j_data_unmatched if "Jet_eta" in col]
# phi_cols_data = [col for col in j_data_unmatched if "Jet_phi" in col]
# pt_cols_data = [col for col in j_data_unmatched if "Jet_pt" in col]

# eta_cols_emb = [col for col in j_emb_unmatched if "Jet_eta" in col]
# phi_cols_emb = [col for col in j_emb_unmatched if "Jet_phi" in col]
# pt_cols_emb = [col for col in j_emb_unmatched if "Jet_pt" in col]

# j_eta_data = j_data_unmatched[eta_cols_data].values.flatten()
# j_phi_data = j_data_unmatched[phi_cols_data].values.flatten()
# j_pt_data = j_data_unmatched[pt_cols_data].values.flatten()

# j_eta_emb = j_emb_unmatched[eta_cols_emb].values.flatten()
# j_phi_emb = j_emb_unmatched[phi_cols_emb].values.flatten()
# j_pt_emb = j_emb_unmatched[pt_cols_emb].values.flatten()


# bins = np.linspace(-5, 5, nbins)
# ax = nq_comparison({"Data jets": j_eta_data, "Emb. jets": j_eta_emb}, bins, r"Unm. jet $\eta$")
# ax.set_yscale("log")
# plt.savefig(os.path.join(output_path, f"control_Jet_eta.png"))
# plt.close()

# bins = np.linspace(-5, 5, nbins)
# ax = nq_comparison({"Data jets": j_phi_data, "Emb. jets": j_phi_emb}, bins, r"Unm. jet $\phi$")
# ax.set_yscale("log")
# plt.savefig(os.path.join(output_path, f"control_Jet_phi.png"))
# plt.close()

# bins = np.linspace(0, 120, nbins)
# ax = nq_comparison({"Data jets": j_pt_data, "Emb. jets": j_pt_emb}, bins, r"Unmatched jet $p_\text{T}/$ GeV")
# ax.set_yscale("log")
# ax.vlines(25, ax.get_ylim()[0], ax.get_ylim()[1], color="black", ls="dashed", linewidth=2)
# plt.savefig(os.path.join(output_path, f"control_Jet_pt.png"))
# plt.close()

In [ ]:
if muon_plots:
    n_muons_data = count_n_objects(data_df, "Muon_eta_")

    dr = calculate_dr(emb_df.copy(deep=True), "muon_all", filter=None, df2=data_df.copy(deep=True))

    muons_data_unmatched, muons_emb_unmatched = find_unmatchable_objects(dr, emb_df.copy(deep=True), data_df.copy(deep=True), "muon_all", muon_cut)

    nlone_muons_data = count_n_objects(muons_data_unmatched, "Muon_eta_")
    nlone_muons_emb = count_n_objects(muons_emb_unmatched, "Muon_eta_")

    n_matchable_muons = n_muons_data - nlone_muons_data
    # n_matchable_muon_emb = n_muons_emb - nlone_muons_emb

    print_stats(n_matchable_muons, nlone_muons_data, nlone_muons_emb)


    muons_data_unmatched = copy_columns_from_to(emb_df, muons_data_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
    muons_emb_unmatched = copy_columns_from_to(emb_df, muons_emb_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
    distances_data, distances_emb = dist_between_zmumu_unmatch(muons_data_unmatched, muons_emb_unmatched, basename="Muon_")

    print_zmumu_stats(distances_data, distances_emb, muon_cut)

In [ ]:
if muon_plots:
    max_nbins = max([np.amax(nlone_muons_data), np.amax(nlone_muons_emb), np.amax(n_matchable_muons)])
    bins = np.arange(-0.5, max_nbins+0.5, 1)
    ax = nq_comparison({"Matched muons": n_matchable_muons, "Unm. muons in data":nlone_muons_data, "Unm. muons in emb":nlone_muons_emb}, bins, "Number of muons")
    ax.set_yscale("log")

    plt.savefig(os.path.join(output_path, "unmatched_muons_00.png"))
    plt.close()

    bins = np.linspace(0,np.amax([np.nanmax(distances_data), np.nanmax(distances_emb)]), 30)
    ax = nq_comparison({"Data distances":distances_data.flatten(), "Embedding distances":distances_emb.flatten()}, bins, r"$\Delta R_\text{min (Z->µµ, unm. µ)}$")
    ax.vlines(muon_cut, ax.get_ylim()[0], ax.get_ylim()[1], color="black", ls="dashed", linewidth=2)
    ax.set_yscale("log")
    plt.savefig(os.path.join(output_path, "dr_zmumu_unmatched_muon_00.png"))
    plt.close()

In [ ]:
# nlone_jets = nlone_jets_data+nlone_jets_emb
# nlone_muons = nlone_muons_data + nlone_muons_emb
# nlone_sum = nlone_jets + nlone_muons

# lm_diff = np.absolute(data_df["LM_pt"]- emb_df["LM_pt"])
# lj_diff = np.absolute(data_df["LJ_pt"]- emb_df["LJ_pt"])
# l_diff = lj_diff + lm_diff

# mpt = np.absolute(data_df["PuppiMET_pt"]- emb_df["PuppiMET_pt"])

# print(spearmanr(mpt, nlone_muons, nan_policy="omit"))
# print(spearmanr(mpt, nlone_jets, nan_policy="omit"))
# print(spearmanr(mpt, nlone_sum, nan_policy="omit"))
# print(spearmanr(mpt, lm_diff, nan_policy="omit"))
# print(spearmanr(mpt, lj_diff, nan_policy="omit"))
# print(spearmanr(mpt, l_diff, nan_policy="omit"))

In [ ]:

# lm_diff = np.absolute(data_df["LM_pt"]+ emb_df["LM_pt"])
# lj_diff = np.absolute(data_df["LJ_pt"]+ emb_df["LJ_pt"])
# l_diff = lj_diff + lm_diff

# mpt = np.absolute(data_df["PuppiMET_pt"]- emb_df["PuppiMET_pt"])

# print(spearmanr(mpt, lm_diff, nan_policy="omit"))
# print(spearmanr(mpt, lj_diff, nan_policy="omit"))
# print(spearmanr(mpt, l_diff, nan_policy="omit"))

In [ ]:
#removing events with unmatchable objects

# print(len(data_df))

# muon_mask = np.logical_and(nlone_jets_data==0, nlone_jets_emb==0)

# jet_mask = np.logical_and(nlone_muons_data==0, nlone_muons_emb==0)

# mask = np.logical_and(muon_mask, jet_mask)

# data_df = data_df.loc[mask]
# emb_df = emb_df.loc[mask]

# print(len(data_df))

In [ ]:
########################################################################################################################################################################
# Applying quality cuts on muons and jets
########################################################################################################################################################################

# emb_df = emb_df[emb_df["nGenPart"]==3]

jet_filters = [
    {"col":"Jet_pt",  "min":25,  "max":None},
    # {"col":"Jet_eta",  "min":3,  "max":None}
]
# electron_filters = [
#     {"col":"Electron_pt",  "min":5,  "max":None},
# ]
# photon_filters = [
#     # {"col":"Photon_pt",  "min":5,  "max":None},
# ]
muon_filters = [
    {"col":"Muon_pt",  "min":8,  "max":None},
    # {"col":"MuonIsGlobal",  "min":0.5,  "max":None},
    {"col":"MuonIsLoose",  "min":0.5,  "max":None}
    # {"col":"MuonIsMedium",  "min":0.5,  "max":None}
    # {"col":"MuonIsTight",  "min":0.5,  "max":None}
]

data_df = quality_cut(data_df, jet_filters, "jet")
emb_df = quality_cut(emb_df, jet_filters, "jet")

data_df = quality_cut(data_df, muon_filters, "muon")
emb_df = quality_cut(emb_df, muon_filters, "muon")

# data_df = quality_cut(data_df, electron_filters, "electron")
# emb_df = quality_cut(emb_df, electron_filters, "electron")

data_df = compactify_objects(data_df, get_jet_basenames(), get_n_occurence(data_df, "Jet_eta_"))
data_df = compactify_objects(data_df, get_muon_basenames(), get_n_occurence(data_df, "Muon_eta_"))
# data_df = compactify_objects(data_df, get_electron_basenames(), get_n_occurence(data_df, "Electron_eta_"))

emb_df = compactify_objects(emb_df, get_jet_basenames(), get_n_occurence(emb_df, "Jet_eta_"))
emb_df = compactify_objects(emb_df, get_muon_basenames(), get_n_occurence(emb_df, "Muon_eta_"))
# emb_df = compactify_objects(emb_df, get_electron_basenames(), get_n_occurence(emb_df, "Electron_eta_"))

data_df, emb_df = create_concordant_subsets(data_df, emb_df)

verify_events(data_df, emb_df)

print(f"Quality cuts applied\n\tLength dataset:\t {len(emb_df)} events")


# data_df = require_min_n(data_df, "Muon_eta_", 2)
# emb_df = require_min_n(emb_df, "Muon_eta_", 2)

# data_df, emb_df = create_concordant_subsets(data_df, emb_df)
# verify_events(data_df, emb_df)

# print(f"Removed events with less than 2 muons\n\tLength dataset:\t {len(emb_df)} events")

In [ ]:
########################################################################################################################################################################
# Applying muon matching
########################################################################################################################################################################

match_filter = [
    # {"col":"dr", "min":-0.05, "max":muon_cut},
    {"col":"LM_pt", "min":16, "max":np.inf},
    {"col":"TM_pt", "min":8, "max":np.inf},
    # {"col":"MuonIsLoose", "min":0.5, "max":np.inf},
]

selection_q_converted = [element["target"] for element in selection_q]
data_df = copy_columns_from_to(emb_df, data_df, selection_q_converted)
emb_df_for_matching = get_matching_df(emb_df, ["LM_pt", "TM_pt", "LM_eta", "TM_eta", "LM_phi", "TM_phi", "LM_m", "TM_m"])


dr = calculate_dr(emb_df, "muon", filter=match_filter)
emb_df, muon_id_matched, dr_matched = apply_genmatching(dr.copy(), emb_df_for_matching.copy(deep=True), "muon")

In [ ]:
########################################################################################################################################################################
# Adding mvis and ptvis
########################################################################################################################################################################

data_df["m_vis"], data_df["pt_vis"] = get_z_m_pt(data_df)
emb_df["m_vis"], emb_df["pt_vis"] = get_z_m_pt(emb_df)

print("Added m_vis and pt_vis")

data_df = data_df.loc[data_df["m_vis"]>18]
emb_df = emb_df.loc[emb_df["m_vis"]>18]

data_df, emb_df = create_concordant_subsets(data_df, emb_df)
verify_events(data_df, emb_df)

print(f"Removed events with m_vis<18. \nLength dataset:\t {len(emb_df)} events")

In [ ]:
if jet_plots:
    n_jets_data = count_n_objects(data_df, "Jet_eta_")
    # n_jets_emb = count_n_objects(emb_df, "Jet_eta_")

    dr = calculate_dr(emb_df.copy(deep=True), "jet_all", filter=None, df2=data_df.copy(deep=True))

    j_data_unmatched, j_emb_unmatched = find_unmatchable_objects(dr, emb_df.copy(deep=True), data_df.copy(deep=True), "jet_all", jet_cut)

    nlone_jets_data = count_n_objects(j_data_unmatched, "Jet_eta_")
    nlone_jets_emb = count_n_objects(j_emb_unmatched, "Jet_eta_")

    n_matchable_jets = n_jets_data - nlone_jets_data
    # n_matchable_jets_emb = n_jets_emb - nlone_jets_emb # is the same

    print_stats(n_matchable_jets, nlone_jets_data, nlone_jets_emb)


    j_data_unmatched = copy_columns_from_to(emb_df, j_data_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
    j_emb_unmatched = copy_columns_from_to(emb_df, j_emb_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
    distances_data, distances_emb = dist_between_zmumu_unmatch(j_data_unmatched, j_emb_unmatched)

    print_zmumu_stats(distances_data, distances_emb, 0.4)

In [ ]:
if jet_plots:
    max_nbins = max([np.amax(nlone_jets_data), np.amax(nlone_jets_emb), np.amax(n_matchable_jets)])
    bins = np.arange(-0.5, max_nbins+0.5, 1)
    ax = nq_comparison({"Matched jets":n_matchable_jets, "Unm. jets in data": nlone_jets_data, "Unm. jets in emb": nlone_jets_emb}, bins, "Number of jets")
    ax.set_yscale("log")

    plt.savefig(os.path.join(output_path, "unmatched_jets_01.png"))
    plt.close()


    bins = np.linspace(0,np.amax([np.nanmax(distances_data), np.nanmax(distances_emb)]), 30)
    ax = nq_comparison({"Data distances":distances_data.flatten(), "Embedding distances":distances_emb.flatten()}, bins, r"$\Delta R_\text{min (Z->µµ, unm. jet)}$")
    ax.vlines(0.4, ax.get_ylim()[0], ax.get_ylim()[1], color="black", ls="dashed", linewidth=2)
    ax.set_yscale("log")
    plt.savefig(os.path.join(output_path, "dr_zmumu_unmatched_jet_01.png"))
    plt.close()

In [ ]:
if muon_plots:
    n_muons_data = count_n_objects(data_df, "Muon_eta_")

    dr = calculate_dr(emb_df.copy(deep=True), "muon_all", filter=None, df2=data_df.copy(deep=True))

    muons_data_unmatched, muons_emb_unmatched = find_unmatchable_objects(dr, emb_df.copy(deep=True), data_df.copy(deep=True), "muon_all", muon_cut)

    nlone_muons_data = count_n_objects(muons_data_unmatched, "Muon_eta_")
    nlone_muons_emb = count_n_objects(muons_emb_unmatched, "Muon_eta_")

    n_matchable_muons = n_muons_data - nlone_muons_data
    # n_matchable_muon_emb = n_muons_emb - nlone_muons_emb

    print_stats(n_matchable_muons, nlone_muons_data, nlone_muons_emb)


    muons_data_unmatched = copy_columns_from_to(emb_df, muons_data_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
    muons_emb_unmatched = copy_columns_from_to(emb_df, muons_emb_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
    distances_data, distances_emb = dist_between_zmumu_unmatch(muons_data_unmatched, muons_emb_unmatched, basename="Muon_")

    print_zmumu_stats(distances_data, distances_emb, muon_cut)

In [ ]:
if muon_plots:
    max_nbins = max([np.amax(nlone_muons_data), np.amax(nlone_muons_emb), np.amax(n_matchable_muons)])
    bins = np.arange(-0.5, max_nbins+0.5, 1)
    ax = nq_comparison({"Matched muons": n_matchable_muons, "Unm. muons in data":nlone_muons_data, "Unm. muons in emb":nlone_muons_emb}, bins, "Number of muons")
    ax.set_yscale("log")

    plt.savefig(os.path.join(output_path, "unmatched_muons_01.png"))
    plt.close()

    bins = np.linspace(0,np.amax([np.nanmax(distances_data), np.nanmax(distances_emb)]), 30)
    ax = nq_comparison({"Data distances":distances_data.flatten(), "Embedding distances":distances_emb.flatten()}, bins, r"$\Delta R_\text{min (Z->µµ, unm. µ)}$")
    ax.vlines(muon_cut, ax.get_ylim()[0], ax.get_ylim()[1], color="black", ls="dashed", linewidth=2)
    ax.set_yscale("log")
    plt.savefig(os.path.join(output_path, "dr_zmumu_unmatched_muon_01.png"))
    plt.close()


In [ ]:
# if electron_plots:
#     n_electrons_data = count_n_objects(data_df, "Electron_eta_")
#     # n_muons_emb = count_n_objects(emb_df, "eta_")

#     electron_matching_df = copy_column_set(data_df.copy(deep=True), emb_df.copy(deep=True), "Electron_")

#     dr = calculate_dr(electron_matching_df, "electron_all", filter=None, df2=data_df.copy(deep=True))

#     electrons_data_unmatched, electrons_emb_unmatched = find_unmatchable_objects(dr, emb_df, data_df, "electron_all", electron_cut)

#     nlone_electrons_data = count_n_objects(electrons_data_unmatched, "Electron_eta_")
#     nlone_electrons_emb = count_n_objects(electrons_emb_unmatched, "Electron_eta_")

#     n_matchable_electrons = n_electrons_data - nlone_electrons_data

#     print_stats(n_matchable_electrons, nlone_electrons_data, nlone_electrons_emb)


#     electrons_data_unmatched = copy_columns_from_to(emb_df, electrons_data_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
#     electrons_emb_unmatched = copy_columns_from_to(emb_df, electrons_emb_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
#     distances_data, distances_emb = dist_between_zmumu_unmatch(electrons_data_unmatched, electrons_emb_unmatched, basename="Electron_")


In [ ]:
# if electron_plots:
#     max_nbins = max([np.amax(nlone_electrons_data), np.amax(nlone_electrons_emb), np.amax(n_matchable_electrons)])
#     bins = np.arange(-0.5, max_nbins+0.5, 1)
#     ax = nq_comparison({"Matched electrons": n_matchable_electrons, "Unm. electrons in data":nlone_electrons_data, "Unm. electrons in emb":nlone_electrons_emb}, bins, "Number of electrons")
#     ax.set_yscale("log")

#     plt.savefig(os.path.join(output_path, "unmatched_electrons_00.png"))
#     plt.close()
#     bins = np.linspace(0,np.amax([np.nanmax(distances_data), np.nanmax(distances_emb)]), 30)
#     ax = nq_comparison({"Data distances":distances_data.flatten(), "Embedding distances":distances_emb.flatten()}, bins, r"$\Delta R_\text{µ1µ2, unm. electron}$")
#     ax.set_yscale("log")
#     plt.savefig(os.path.join(output_path, "dr_zmumu_unmatched_electron_00.png"))
#     plt.close()

In [ ]:
# if photon_plots:
#     n_photons_data = count_n_objects(data_df, "Photon_eta_")

#     dr = calculate_dr(emb_df.copy(deep=True), "photon_all", filter=None, df2=data_df.copy(deep=True))
#     # dr_safe = dr.copy()
#     photons_data_unmatched, photons_emb_unmatched = find_unmatchable_objects(dr, emb_df, data_df, "photon_all", photon_cut)

#     nlone_photons_data = count_n_objects(photons_data_unmatched, "Photon_eta_")
#     nlone_photons_emb = count_n_objects(photons_emb_unmatched, "Photon_eta_")

#     n_matchable_photons = n_photons_data - nlone_photons_data

#     print_stats(n_matchable_photons, nlone_photons_data, nlone_photons_emb)


#     photons_data_unmatched = copy_columns_from_to(emb_df, photons_data_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
#     photons_emb_unmatched = copy_columns_from_to(emb_df, photons_emb_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
#     distances_data, distances_emb = dist_between_zmumu_unmatch(photons_data_unmatched, photons_emb_unmatched, basename="Photon_")


In [ ]:
# if photon_plots:
#     max_nbins = max([np.amax(nlone_photons_data), np.amax(nlone_photons_emb), np.amax(n_matchable_photons)])
#     bins = np.arange(-0.5, max_nbins+0.5, 1)
#     ax = nq_comparison({"Matched photons": n_matchable_photons, "Unm. photons in data":nlone_photons_data, "Unm. photons in emb":nlone_photons_emb}, bins, "Number of photons")
#     ax.set_yscale("log")

#     plt.savefig(os.path.join(output_path, "unmatched_photons_00.png"))
#     plt.close()

#     bins = np.linspace(0,np.amax([np.nanmax(distances_data), np.nanmax(distances_emb)]), 30)
#     ax = nq_comparison({"Data distances":distances_data.flatten(), "Embedding distances":distances_emb.flatten()}, bins, r"$\Delta R_\text{µ1µ2, unm. photon}$")
#     ax.set_yscale("log")
#     plt.savefig(os.path.join(output_path, "dr_zmumu_unmatched_photon_00.png"))
#     plt.close()

In [ ]:
########################################################################################################################################################################
# Removing zmumu candidates
########################################################################################################################################################################

# nmuon_total_data = count_n_objects(data_df, "Muon_eta_")
# nmuon_total_emb = count_n_objects(emb_df, "Muon_eta_")

# # dr_data = calculate_dr(data_df, "muon", filter=None)
# # # _, _, dr_data_matched = apply_genmatching(dr_data.copy(), data_df.copy(deep=True), "muon")

# # emb_df_copy = copy_columns_from_to(data_df, emb_df.copy(deep=True), ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
# # dr_emb = calculate_dr(emb_df_copy, "muon", filter=None)
# # # emb_df_copy = get_matching_df(emb_df_copy, ["LM_pt", "TM_pt", "LM_eta", "TM_eta", "LM_phi", "TM_phi", "LM_m", "TM_m"])
# # # _, _, dr_emb_matched = apply_genmatching(dr_emb.copy(), emb_df_copy, "muon")

# muons_data = get_sub_df(data_df.copy(deep=True), "Muon_")
# muons_emb = get_sub_df(emb_df.copy(deep=True), "Muon_")


# muons_data = copy_columns_from_to(data_df, muons_data, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
# muons_emb = copy_columns_from_to(data_df, muons_emb, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])

# distances_data, distances_emb = dist_between_zmumu_unmatch(muons_data, muons_emb, "Muon_")
# n_cands_data = np.sum(distances_data<muon_cut)
# print(f"number of obj in zmumu proximity: {n_cands_data}")
# n_cands_emb = np.sum(distances_emb<muon_cut)
# print(f"number of obj in zmumu proximity: {n_cands_emb}")


# data_df = remove_zmumu_candidates(data_df, distances_data, muon_cut, "first")
# emb_df = remove_zmumu_candidates(emb_df, distances_emb, muon_cut, "first")

# data_df = remove_zmumu_candidates(data_df, distances_data, muon_cut)
# emb_df = remove_zmumu_candidates(emb_df, distances_emb, muon_cut)


# nmuon_cleaned_data = count_n_objects(data_df, "Muon_eta_")
# nmuon_cleaned_emb = count_n_objects(emb_df, "Muon_eta_")


# val1 = np.sum(nmuon_total_data)-np.sum(nmuon_cleaned_data)
# val2 = np.sum(nmuon_total_emb)-np.sum(nmuon_cleaned_emb)

# print(f"Removed {val1} muons from {len(data_df)} data events")
# print(f"Removed {val2} muons from {len(emb_df)} emb events")

# # print(len(data_df), len(emb_df))

# data_df = compactify_objects(data_df, get_muon_basenames(), get_n_occurence(data_df, "Muon_eta_"))
# # data_df = compactify_objects(data_df, get_muon_basenames(), get_n_occurence(data_df, "eta_"))

# emb_df = compactify_objects(emb_df, get_muon_basenames(), get_n_occurence(emb_df, "Muon_eta_"))
# # emb_df = compactify_objects(emb_df, get_muon_basenames(), get_n_occurence(emb_df, "eta_"))


# emb_df = require_min_n(emb_df, "LM_eta", 1)
# emb_df = require_min_n(emb_df, "TM_eta", 1)

# data_df, emb_df = create_concordant_subsets(data_df, emb_df)
# verify_events(data_df, emb_df)

# print(f"Removed events with less than 2 zmumu candidates:\t {len(emb_df)} events")

In [ ]:

########################################################################################################################################################################
# Removing muon jets
########################################################################################################################################################################



njet_total_emb = count_n_objects(emb_df, "Jet_eta_")
njet_total_data = count_n_objects(data_df, "Jet_eta_")

# dr1 = calculate_dr(data_df, "filter", filter=None)
# data_df = remove_muon_jets(data_df, dr1, jet_cut)

emb_df_copy = copy_columns_from_to(data_df, emb_df.copy(deep=True), ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
# dr2 = calculate_dr(emb_df_copy, "filter", filter=None)
# emb_df = remove_muon_jets(emb_df, dr2, jet_cut)


distances_data, distances_emb = dist_between_zmumu_unmatch(data_df.copy(deep=True), emb_df_copy)

# data_df = remove_non_muon_jets(data_df, distances_data, jet_cut)
# emb_df = remove_non_muon_jets(emb_df, distances_emb, jet_cut)

data_df = remove_muon_jets(data_df, distances_data, jet_cut)
emb_df = remove_muon_jets(emb_df, distances_emb, jet_cut)

njet_cleaned_emb = count_n_objects(emb_df, "Jet_eta_")
njet_cleaned_data = count_n_objects(data_df, "Jet_eta_")


val1 = np.sum(njet_total_data)-np.sum(njet_cleaned_data)
val2 = np.sum(njet_total_emb)-np.sum(njet_cleaned_emb)

print(f"Removed {val1} jets from {len(data_df)} data events")
print(f"Removed {val2} jets from {len(emb_df)} emb events")

# print(len(data_df), len(emb_df))

data_df = compactify_objects(data_df, get_jet_basenames(), get_n_occurence(data_df, "Jet_eta_"))
# data_df = compactify_objects(data_df, get_muon_basenames(), get_n_occurence(data_df, "eta_"))

emb_df = compactify_objects(emb_df, get_jet_basenames(), get_n_occurence(emb_df, "Jet_eta_"))
# emb_df = compactify_objects(emb_df, get_muon_basenames(), get_n_occurence(emb_df, "eta_"))

data_df, emb_df = create_concordant_subsets(data_df, emb_df)

# print(len(data_df), len(emb_df))


In [ ]:
########################################################################################################################################################################
# Matching jets
########################################################################################################################################################################


data_df, emb_df_for_matching = prepare_matching(data_df, emb_df, "jet")

# match_filter = [
#     {"col":"dr", "min":0, "max":0.1}
# ]

dr = calculate_dr(emb_df_for_matching, "jet", filter=None)

emb_df, jet_id_matched, jet_dr_matched = apply_genmatching(dr.copy(), emb_df, "jet")

# print("jet_match_dr>0.2", np.sum(jet_dr_matched[:,0]>0.2), np.sum(~np.isnan(jet_dr_matched[:,0])))
# print("jet_match_dr2", np.sum(jet_dr_matched[:,1]>0.2), np.sum(~np.isnan(jet_dr_matched[:,1])))
# print("jet_match_dr<0.2", np.sum(jet_dr_matched[:,0]<0.2), np.sum(~np.isnan(jet_dr_matched[:,0])))
# print("jet_match_dr2", np.sum(jet_dr_matched[:,1]<0.2), np.sum(~np.isnan(jet_dr_matched[:,1])))
# print("jet_match_dr-0.4", np.sum(jet_dr_matched[:,0]>0.4), np.sum(~np.isnan(jet_dr_matched[:,0])))
# print("jet_match_dr2", np.sum(jet_dr_matched[:,1]>0.4), np.sum(~np.isnan(jet_dr_matched[:,1])))

data_df, emb_df = remove_nonmatches(data_df, emb_df, "jet")


print("Jets matched")

In [ ]:
# store = pd.HDFStore(os.path.join(output_path, "converted_nanoaod.h5"), 'w')  
# store.put("data_df", data_df, index=False)
# store.put("emb_df", emb_df, index=False)
# store.close()

# print("Data stored in hdf store")

In [ ]:
########################################################################################################################################################################
# Matching electrons
########################################################################################################################################################################

# data_df, emb_df_for_matching = prepare_matching(data_df, emb_df, "electron")

# # match_filter = [
# #     {"col":"dr", "min":0, "max":0.1}
# # ]

# dr = calculate_dr(emb_df_for_matching, "electron", filter=None)

# emb_df, electron_id_matched, electron_dr_matched = apply_genmatching(dr.copy(), emb_df, "electron")


# data_df, emb_df = remove_nonmatches(data_df, emb_df, "electron")


# print("Electrons matched")


In [ ]:
########################################################################################################################################################################
# Matching Photons
########################################################################################################################################################################

# data_df, emb_df_for_matching = prepare_matching(data_df, emb_df, "photon")

# # match_filter = [
# #     {"col":"dr", "min":0, "max":0.1}
# # ]

# dr = calculate_dr(emb_df_for_matching, "photon", filter=None)

# emb_df, electron_id_matched, electron_dr_matched = apply_genmatching(dr.copy(), emb_df, "photon")


# data_df, emb_df = remove_nonmatches(data_df, emb_df, "photon")


# print("Photons matched")

In [ ]:
if jet_plots:
    n_jets_data = count_n_objects(data_df, "Jet_eta_")
    # n_jets_emb = count_n_objects(emb_df, "Jet_eta_")

    dr = calculate_dr(emb_df.copy(deep=True), "jet_all", filter=None, df2=data_df.copy(deep=True))


    j_data_unmatched, j_emb_unmatched = find_unmatchable_objects(dr.copy(), emb_df, data_df, "jet_all", jet_cut)

    nlone_jets_data = count_n_objects(j_data_unmatched, "Jet_eta_")
    nlone_jets_emb = count_n_objects(j_emb_unmatched, "Jet_eta_")

    n_matchable_jets = n_jets_data - nlone_jets_data
    # n_matchable_jets_emb = n_jets_emb - nlone_jets_emb # is the same

    print_stats(n_matchable_jets, nlone_jets_data, nlone_jets_emb)


    j_data_unmatched = copy_columns_from_to(data_df, j_data_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
    j_emb_unmatched = copy_columns_from_to(data_df, j_emb_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
    distances_data, distances_emb = dist_between_zmumu_unmatch(j_data_unmatched, j_emb_unmatched)
    print_zmumu_stats(distances_data, distances_emb, 0.4)

In [ ]:
if jet_plots:
    max_nbins = max([np.amax(nlone_jets_data), np.amax(nlone_jets_emb), np.amax(n_matchable_jets)])
    bins = np.arange(-0.5, max_nbins+2.5, 1)
    ax = nq_comparison({"Matched jets":n_matchable_jets, "Unm. jets in data": nlone_jets_data, "Unm. jets in emb": nlone_jets_emb}, bins, "Number of jets")
    ax.set_yscale("log")

    plt.savefig(os.path.join(output_path, "unmatched_jets_02.png"))
    plt.close() 


    bins = np.linspace(0,np.amax([np.nanmax(distances_data), np.nanmax(distances_emb)]), 30)
    ax = nq_comparison({"Data distances":distances_data.flatten(), "Embedding distances":distances_emb.flatten()}, bins, r"$\Delta R_\text{min (Z->µµ, unm. jet)}$")
    ax.vlines(0.4, ax.get_ylim()[0], ax.get_ylim()[1], color="black", ls="dashed", linewidth=2)
    ax.set_yscale("log")
    plt.savefig(os.path.join(output_path, "dr_zmumu_unmatched_jet_02.png"))
    plt.close()

In [ ]:
if muon_plots:
    n_muons_data = count_n_objects(data_df, "Muon_eta_")
    # n_muons_emb = count_n_objects(emb_df, "eta_")

    dr = calculate_dr(emb_df.copy(deep=True), "muon_all", filter=None, df2=data_df.copy(deep=True))

    muons_data_unmatched, muons_emb_unmatched = find_unmatchable_objects(dr, emb_df, data_df, "muon_all", muon_cut)

    nlone_muons_data = count_n_objects(muons_data_unmatched, "Muon_eta_")
    nlone_muons_emb = count_n_objects(muons_emb_unmatched, "Muon_eta_")

    n_matchable_muons = n_muons_data - nlone_muons_data
    # n_matchable_muon_emb = n_muons_emb - nlone_muons_emb

    print_stats(n_matchable_muons, nlone_muons_data, nlone_muons_emb)


    muons_data_unmatched = copy_columns_from_to(data_df, muons_data_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
    muons_emb_unmatched = copy_columns_from_to(data_df, muons_emb_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])

    distances_data, distances_emb = dist_between_zmumu_unmatch(muons_data_unmatched, muons_emb_unmatched, "Muon_")

    print_zmumu_stats(distances_data, distances_emb, muon_cut)

In [ ]:
if muon_plots:
    max_nbins = max([np.amax(nlone_muons_data), np.amax(nlone_muons_emb), np.amax(n_matchable_muons)])
    bins = np.arange(-0.5, max_nbins+0.5, 1)
    ax = nq_comparison({"Matched muons": n_matchable_muons, "Unm. muons in data":nlone_muons_data, "Unm. muons in emb":nlone_muons_emb}, bins, "Number of muons")
    ax.set_yscale("log")

    plt.savefig(os.path.join(output_path, "unmatched_muons_02.png"))
    plt.close()


    bins = np.linspace(0,np.amax([np.nanmax(distances_data), np.nanmax(distances_emb)]), 30)
    ax = nq_comparison({"Data distances":distances_data.flatten(), "Embedding distances":distances_emb.flatten()}, bins, r"$\Delta R_\text{min (Z->µµ, unm. µ)}$")
    ax.vlines(muon_cut, ax.get_ylim()[0], ax.get_ylim()[1], color="black", ls="dashed", linewidth=2)
    ax.set_yscale("log")
    plt.savefig(os.path.join(output_path, "dr_zmumu_unmatched_muon_02.png"))
    plt.close()

In [ ]:
# nlone_jets = nlone_jets_data+nlone_jets_emb
# nlone_muons = nlone_muons_data + nlone_muons_emb
# nlone_sum = nlone_jets + nlone_muons

# lj_diff = np.absolute(data_df["LJ_pt"]- emb_df["LJ_pt"])
# lm_diff = np.absolute(data_df["LM_pt"]- emb_df["LM_pt"])
# l_diff = lj_diff + lm_diff

# mpt = np.absolute(data_df["PuppiMET_pt"]- emb_df["PuppiMET_pt"])

# print(spearmanr(mpt, nlone_jets, nan_policy="omit"))
# print(spearmanr(mpt, nlone_muons, nan_policy="omit"))
# print(spearmanr(mpt, nlone_sum, nan_policy="omit"))
# print(spearmanr(mpt, lj_diff, nan_policy="omit"))
# print(spearmanr(mpt, lm_diff, nan_policy="omit"))
# print(spearmanr(mpt, l_diff, nan_policy="omit"))

In [ ]:
# if electron_plots:
#     n_electrons_data = count_n_objects(data_df, "Electron_eta_")
#     # n_muons_emb = count_n_objects(emb_df, "eta_")

#     electron_matching_df = copy_column_set(data_df.copy(deep=True), emb_df.copy(deep=True), "Electron_")

#     dr = calculate_dr(electron_matching_df, "electron_all", filter=None, df2=data_df.copy(deep=True))

#     electrons_data_unmatched, electrons_emb_unmatched = find_unmatchable_objects(dr, emb_df, data_df, "electron_all", electron_cut)

#     nlone_electrons_data = count_n_objects(electrons_data_unmatched, "Electron_eta_")
#     nlone_electrons_emb = count_n_objects(electrons_emb_unmatched, "Electron_eta_")

#     n_matchable_electrons = n_electrons_data - nlone_electrons_data

#     print_stats(n_matchable_electrons, nlone_electrons_data, nlone_electrons_emb)

#     electrons_data_unmatched = copy_columns_from_to(data_df, electrons_data_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
#     electrons_emb_unmatched = copy_columns_from_to(data_df, electrons_emb_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
#     distances_data, distances_emb = dist_between_zmumu_unmatch(electrons_data_unmatched, electrons_emb_unmatched, "Electron_")

In [ ]:
# if electron_plots:
#     max_nbins = max([np.amax(nlone_electrons_data), np.amax(nlone_electrons_emb), np.amax(n_matchable_electrons)])
#     bins = np.arange(-0.5, max_nbins+0.5, 1)
#     ax = nq_comparison({"Matched electrons": n_matchable_electrons, "Unm. electrons in data":nlone_electrons_data, "Unm. electrons in emb":nlone_electrons_emb}, bins, "Number of electrons")
#     ax.set_yscale("log")

#     plt.savefig(os.path.join(output_path, "unmatched_electrons_02.png"))
#     plt.close()

#     bins = np.linspace(0,np.amax([np.nanmax(distances_data), np.nanmax(distances_emb)]), 30)
#     ax = nq_comparison({"Data distances":distances_data.flatten(), "Embedding distances":distances_emb.flatten()}, bins, r"$\Delta R_\text{µ1µ2, unm. electron}$")
#     ax.set_yscale("log")
#     plt.savefig(os.path.join(output_path, "dr_zmumu_unmatched_electron_02.png"))
#     plt.close()

In [ ]:
# if photon_plots:
#     n_photons_data = count_n_objects(data_df, "Photon_eta_")

#     dr = calculate_dr(emb_df.copy(deep=True), "photon_all", filter=None, df2=data_df.copy(deep=True))
#     # dr_safe = dr.copy()
#     photons_data_unmatched, photons_emb_unmatched = find_unmatchable_objects(dr, emb_df, data_df, "photon_all", photon_cut)

#     nlone_photons_data = count_n_objects(photons_data_unmatched, "Photon_eta_")
#     nlone_photons_emb = count_n_objects(photons_emb_unmatched, "Photon_eta_")

#     n_matchable_photons = n_photons_data - nlone_photons_data

#     print_stats(n_matchable_photons, nlone_photons_data, nlone_photons_emb)

#     photons_data_unmatched = copy_columns_from_to(data_df, photons_data_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
#     photons_emb_unmatched = copy_columns_from_to(data_df, photons_emb_unmatched, ["LM_pt", "LM_eta", "LM_phi", "TM_pt", "TM_eta", "TM_phi"])
#     distances_data, distances_emb = dist_between_zmumu_unmatch(photons_data_unmatched, photons_emb_unmatched, "Photon_")

In [ ]:
# if photon_plots:
#     max_nbins = max([np.amax(nlone_photons_data), np.amax(nlone_photons_emb), np.amax(n_matchable_photons)])
#     bins = np.arange(-0.5, max_nbins+0.5, 1)
#     ax = nq_comparison({"Matched photons": n_matchable_photons, "Unm. photons in data":nlone_photons_data, "Unm. photons in emb":nlone_photons_emb}, bins, "Number of photons")
#     ax.set_yscale("log")

#     plt.savefig(os.path.join(output_path, "unmatched_photons_02.png"))
#     plt.close()

#     bins = np.linspace(0,np.amax([np.nanmax(distances_data), np.nanmax(distances_emb)]), 30)
#     ax = nq_comparison({"Data distances":distances_data.flatten(), "Embedding distances":distances_emb.flatten()}, bins, r"$\Delta R_\text{µ1µ2, unm. photon}$")
#     ax.set_yscale("log")
#     plt.savefig(os.path.join(output_path, "dr_zmumu_unmatched_photon_02.png"))
#     plt.close()